In [1]:
# Emerging Technologies — Problems

In [2]:
## Problem 1: Generating Random Boolean Functions

This problem asks for a Python function `random_constant_balanced` that returns a
randomly chosen function from the set of **constant** or **balanced** Boolean
functions taking four Boolean arguments as input.

### Background

The Deutsch–Jozsa algorithm [1] is one of the earliest examples of a quantum
algorithm that provides a provable speedup over deterministic classical
algorithms. It is designed to decide, with a single query, whether a given
Boolean function $f : \{0,1\}^n \to \{0,1\}$ belongs to one of two restricted
classes:

- **Constant functions** — $f(x) = c$ for every input $x$, where $c \in \{0,1\}$.
- **Balanced functions** — $f$ returns $0$ on exactly half of the $2^n$ possible
  inputs and $1$ on the other half.

A general Boolean function need not be either of these; in fact, most are
neither. The Deutsch–Jozsa problem is restricted *by promise* to only these two
classes. The modern textbook formulation of the algorithm given by Cleve,
Ekert, Macchiavello and Mosca [2] is the one used in most quantum-computing
tutorials today, including the IBM Quantum Learning material [3] linked in the
problem statement.

Before we can simulate or analyse the algorithm in later problems, we need a
way to generate such functions at random — that is the goal of Problem 1.

### References

[1] D. Deutsch and R. Jozsa, "Rapid solution of problems by quantum
computation," *Proceedings of the Royal Society A*, vol. 439, no. 1907,
pp. 553–558, 1992. https://doi.org/10.1098/rspa.1992.0167

[2] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum algorithms
revisited," *Proceedings of the Royal Society A*, vol. 454, no. 1969,
pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

[3] IBM Quantum Learning, "The Deutsch–Jozsa algorithm."
https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa

SyntaxError: invalid character '–' (U+2013) (2493897688.py, line 9)

In [5]:
### Counting constant and balanced functions on four inputs

A Boolean function $f : \{0,1\}^4 \to \{0,1\}$ is fully specified by its
**truth table** — the list of outputs for each of the $2^4 = 16$ possible
input combinations. Since each of those 16 outputs is independently $0$ or
$1$, there are $2^{16} = 65{,}536$ Boolean functions on four inputs in total
[1].

We are interested in two specific subsets:

**Constant functions.** There are exactly **two**: the function that returns
$0$ everywhere, and the function that returns $1$ everywhere.

**Balanced functions.** A balanced function returns $1$ on exactly half of
the $2^4 = 16$ inputs, and $0$ on the other half. The number of such
functions is therefore the number of ways to choose which 8 of the 16 input
combinations map to $1$:

$$
\binom{16}{8} = 12{,}870.
$$

So the *promise set* for the four-input Deutsch–Jozsa problem contains
$2 + 12{,}870 = 12{,}872$ functions in total. Out of all $65{,}536$ Boolean
functions on four inputs, only about **19.6%** satisfy the Deutsch–Jozsa
promise — the rest are neither constant nor balanced and the algorithm is
not defined for them [2].

This count drives the implementation strategy: rather than rejection-sampling
from all $2^{16}$ functions (which would discard roughly four out of every
five candidates), we will sample directly from the constant and balanced
classes in proportion to their sizes.

### References

[1] D. E. Knuth, *The Art of Computer Programming, Volume 4A: Combinatorial
Algorithms, Part 1*. Upper Saddle River, NJ: Addison-Wesley, 2011, ch. 7.1.1.

[2] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.3.

SyntaxError: invalid syntax (3915817419.py, line 3)

In [6]:
### Representation strategy: truth tables as integers

A Boolean function on four inputs is fully described by its 16-entry truth
table. There are several reasonable ways to represent such a table in
Python, but the most compact is a single 16-bit integer where bit $i$
holds the output $f(x)$ for the input $x$ whose binary encoding is $i$.

For example, the integer `0b1010101010101010` (which is `0xAAAA`, or
$43{,}690$ in decimal) encodes the function

$$
f(x_3, x_2, x_1, x_0) = x_0,
$$

because its truth table outputs $1$ exactly when the least significant
input bit is $1$. This is a balanced function: half of the 16 inputs have
$x_0 = 1$.

This representation has three advantages relevant to our task [1]:

1. **Constant functions are trivial to construct.** The all-zeros function
   is the integer $0$ and the all-ones function is $2^{16} - 1 = 65{,}535$.
2. **Balancedness is a single popcount.** A function is balanced on four
   inputs if and only if its 16-bit truth table has exactly 8 bits set,
   which Python exposes directly as `int.bit_count()` (added in Python 3.10).
3. **Sampling a balanced function reduces to a random bit selection.** We
   can pick which 8 of the 16 input positions map to $1$ uniformly at
   random using `random.sample`, then assemble the integer.

The actual *callable* the user receives will wrap this integer in a closure
that takes four Boolean arguments, packs them into the index $i$, and
returns the corresponding bit of the truth table.

### References

[1] H. S. Warren Jr., *Hacker's Delight*, 2nd ed. Upper Saddle River, NJ:
Addison-Wesley, 2013, ch. 5 ("Counting Bits").

SyntaxError: unterminated string literal (detected at line 36) (1074967894.py, line 36)

In [8]:
"""Imports and helpers for Problem 1."""

import random
from typing import Callable

# A Boolean function on four inputs maps four bools to a single bool.
BoolFunc4 = Callable[[bool, bool, bool, bool], bool]


def _truth_table_to_callable(table: int) -> BoolFunc4:
    """Wrap a 16-bit truth table integer in a 4-argument Boolean callable.

    Bit ``i`` of ``table`` is the output of the function on the input whose
    binary encoding (with ``x3`` as the most significant bit) equals ``i``.

    Parameters
    ----------
    table : int
        A non-negative integer in the range ``[0, 2**16)`` whose binary
        representation is the truth table of the function.

    Returns
    -------
    Callable[[bool, bool, bool, bool], bool]
        A function ``f(x3, x2, x1, x0)`` that returns the corresponding
        truth-table bit as a Python ``bool``.
    """
    if not 0 <= table < (1 << 16):
        raise ValueError(
            f"truth table must fit in 16 bits, got {table}"
        )

    def f(x3: bool, x2: bool, x1: bool, x0: bool) -> bool:
        # Pack the four input bits into an index in [0, 16).
        index = (int(bool(x3)) << 3) | (int(bool(x2)) << 2) \
                | (int(bool(x1)) << 1) | int(bool(x0))
        return bool((table >> index) & 1)

    return f

In [10]:
### Implementing `random_constant_balanced`

We now have everything we need: a representation (16-bit truth table), a
wrapper that turns such a table into a callable, and the counts of each
class (2 constant functions, $\binom{16}{8} = 12{,}870$ balanced functions).

The function below samples uniformly from the union of these two sets. To
keep the sampling truly uniform across all $12{,}872$ functions we weight
the choice between the two branches by their class sizes — otherwise a
50/50 coin flip between "constant" and "balanced" would massively
over-represent the two constant functions.

Within each branch, the sampling is straightforward:

- **Constant branch.** Choose the all-zeros or all-ones truth table with
  equal probability.
- **Balanced branch.** Choose 8 of the 16 input positions uniformly at
  random using `random.sample`, then set those bits in the truth table.

SyntaxError: invalid syntax (4183022430.py, line 3)

In [12]:
def random_constant_balanced(rng: random.Random | None = None) -> BoolFunc4:
    """Return a uniformly random constant or balanced Boolean function on 4 inputs.

    The returned callable accepts four Boolean arguments and returns a single
    Boolean output. The function is drawn uniformly at random from the union
    of:

    - the 2 constant functions on 4 inputs (always ``False``, always ``True``), and
    - the C(16, 8) = 12,870 balanced functions on 4 inputs.

    The branch (constant vs. balanced) is selected with probability proportional
    to the size of each class, so every one of the 12,872 functions in the
    promise set is equally likely to be returned.

    Parameters
    ----------
    rng : random.Random, optional
        A random number generator. If ``None``, the module-level ``random``
        functions are used. Passing an explicit ``random.Random`` instance
        allows the caller to seed the sampling for reproducibility.

    Returns
    -------
    Callable[[bool, bool, bool, bool], bool]
        A Boolean function that is either constant or balanced.
    """
    # Use the supplied generator, or fall back to the module-level one.
    choice = rng.choice if rng is not None else random.choice
    sample = rng.sample if rng is not None else random.sample
    random_func = rng.random if rng is not None else random.random

    # There are 2 constant functions and C(16, 8) = 12,870 balanced functions.
    # Probability of the constant branch = 2 / 12,872.
    n_constant = 2
    n_balanced = 12_870
    p_constant = n_constant / (n_constant + n_balanced)

    if random_func() < p_constant:
        # Constant branch: 0x0000 (always False) or 0xFFFF (always True).
        table = choice([0x0000, 0xFFFF])
    else:
        # Balanced branch: pick 8 of the 16 input positions to map to True.
        ones_positions = sample(range(16), 8)
        table = 0
        for pos in ones_positions:
            table |= (1 << pos)

    return _truth_table_to_callable(table)

In [13]:
### Verifying the implementation

Before using `random_constant_balanced` anywhere else, we should confirm that
it actually upholds the promise. Two checks are appropriate:

1. **Per-sample check.** Every function returned must be either constant
   (all 16 outputs equal) or balanced (exactly 8 outputs are `True`). We
   evaluate the function on all 16 inputs and verify this directly.

2. **Distributional check.** Over many samples, the empirical proportion of
   constant functions should approach the theoretical value
   $2 / 12{,}872 \approx 0.0155\%$. Because constants are so rare, a small
   sample size will not detect bias reliably; we use a large sample and
   allow a generous tolerance.

These are written as plain `assert` statements rather than as a `pytest`
suite so the notebook stays self-contained and reproducible by anyone who
clones the repository, but the structure mirrors what a `pytest` test
module would look like.

SyntaxError: invalid syntax (1725258481.py, line 3)

In [14]:
def _evaluate_truth_table(f: BoolFunc4) -> list[bool]:
    """Evaluate a 4-input Boolean function on all 16 inputs.

    Returns the outputs in index order, where index ``i`` corresponds to
    the input ``(x3, x2, x1, x0)`` whose binary encoding equals ``i``.
    """
    outputs = []
    for i in range(16):
        x3 = bool((i >> 3) & 1)
        x2 = bool((i >> 2) & 1)
        x1 = bool((i >> 1) & 1)
        x0 = bool(i & 1)
        outputs.append(f(x3, x2, x1, x0))
    return outputs


def _classify(f: BoolFunc4) -> str:
    """Return 'constant', 'balanced', or 'neither' for a 4-input function."""
    outputs = _evaluate_truth_table(f)
    n_true = sum(outputs)
    if n_true == 0 or n_true == 16:
        return "constant"
    if n_true == 8:
        return "balanced"
    return "neither"


# --- Per-sample check ---------------------------------------------------------
# Every function returned must be either constant or balanced.
rng = random.Random(20260501)  # seeded for reproducibility of this check
for _ in range(1_000):
    f = random_constant_balanced(rng=rng)
    assert _classify(f) in {"constant", "balanced"}, \
        "random_constant_balanced returned a function that is neither"

print("Per-sample check passed: 1,000 sampled functions were all constant or balanced.")

Per-sample check passed: 1,000 sampled functions were all constant or balanced.


In [15]:
# --- Distributional check -----------------------------------------------------
# Over many samples, the empirical fraction of constant functions should be
# close to 2 / 12_872 ≈ 0.000155.
rng = random.Random(20260502)
n_samples = 200_000
n_constant_observed = 0
for _ in range(n_samples):
    f = random_constant_balanced(rng=rng)
    if _classify(f) == "constant":
        n_constant_observed += 1

expected_p = 2 / 12_872
observed_p = n_constant_observed / n_samples

# The standard deviation of a Binomial(n, p) proportion is sqrt(p(1-p)/n).
# For n=200_000 and p=0.000155, sigma ≈ 2.8e-5, so a 5-sigma tolerance is
# generous and avoids flaky test failures while still catching gross bias.
import math
sigma = math.sqrt(expected_p * (1 - expected_p) / n_samples)
tolerance = 5 * sigma

assert abs(observed_p - expected_p) < tolerance, (
    f"Empirical constant proportion {observed_p:.6f} differs from expected "
    f"{expected_p:.6f} by more than 5 sigma ({tolerance:.6f})."
)

print(f"Distributional check passed:")
print(f"  Expected fraction of constants: {expected_p:.6f}")
print(f"  Observed fraction of constants: {observed_p:.6f}")
print(f"  Tolerance (5σ):                 {tolerance:.6f}")

Distributional check passed:
  Expected fraction of constants: 0.000155
  Observed fraction of constants: 0.000135
  Tolerance (5σ):                 0.000139


In [16]:
### Demonstration

To make the output of `random_constant_balanced` concrete, we draw a few
sample functions and print their truth tables alongside their classification.
A truth table is shown as the 16 outputs in index order, where the index
$i$ corresponds to the input $(x_3, x_2, x_1, x_0)$ whose binary encoding
equals $i$ (so index 0 is `(F, F, F, F)` and index 15 is `(T, T, T, T)`).

Because the constant class contains only 2 of the 12,872 promise functions,
the vast majority of samples will be balanced — drawing a constant by chance
in a small sample is unlikely. To exhibit one of each, the demonstration
below draws random samples until it has shown at least one constant and one
balanced function, capping the search to keep the cell deterministic in
runtime.

SyntaxError: invalid character '—' (U+2014) (974383832.py, line 10)

In [19]:
def _format_truth_table(outputs: list[bool]) -> str:
    """Render a 16-entry truth table as a compact string of 0s and 1s."""
    return "".join("1" if b else "0" for b in outputs)


# Draw samples until we have shown one constant and one balanced function,
# or until we hit a hard cap on the number of attempts.
rng = random.Random(20260503)
shown_constant = False
shown_balanced = False
attempts = 0
max_attempts = 50_000

print(f"{'attempt':>7}  {'class':>9}  truth table (index 0 -> 15)")
print("-" * 52)

while not (shown_constant and shown_balanced) and attempts < max_attempts:
    attempts += 1
    f = random_constant_balanced(rng=rng)
    outputs = _evaluate_truth_table(f)
    label = _classify(f)

    # Only print the first constant and the first few balanced samples,
    # so the cell output stays readable.
    if label == "constant" and not shown_constant:
        print(f"{attempts:>7}  {label:>9}  {_format_truth_table(outputs)}")
        shown_constant = True
    elif label == "balanced" and not shown_balanced:
        print(f"{attempts:>7}  {label:>9}  {_format_truth_table(outputs)}")
        shown_balanced = True

print()
print(f"Found a constant function after {attempts} draws "
      f"(expected ≈ {12_872 // 2} on average).")

attempt      class  truth table (index 0 -> 15)
----------------------------------------------------
      1   balanced  1100000011101110
    384   constant  0000000000000000

Found a constant function after 384 draws (expected ≈ 6436 on average).


In [ ]:
### Summary of Problem 1

We have:

- characterised the promise set of the four-input Deutsch–Jozsa problem
  (12,872 functions: 2 constant, 12,870 balanced);
- chosen a compact 16-bit-integer representation of truth tables that makes
  both construction and verification simple;
- implemented `random_constant_balanced`, which samples uniformly from the
  promise set by weighting the constant and balanced branches by their
  class sizes;
- verified per-sample correctness (every output is constant or balanced)
  and distributional correctness (the empirical fraction of constants
  matches the theoretical $2/12{,}872$ within 5σ over 200,000 samples);
- demonstrated the output on concrete examples.

In the next problem we will use this function to supply the oracle for a
Qiskit implementation of the Deutsch–Jozsa algorithm and compare its
single-query quantum decision against the worst-case classical query
count.

In [ ]:
## Problem 2: Classical Testing for Function Type

This problem asks us to write a Python function `determine_constant_balanced`
that takes a function `f` of the kind generated in Problem 1 (a four-input
Boolean function that is *promised* to be either constant or balanced) and
returns the string `"constant"` or `"balanced"` according to which class `f`
belongs to. We are also asked to comment on the efficiency of the solution
and to state the worst-case number of calls to `f` required for absolute
certainty.

### Background

Establishing the **classical query complexity** of a problem — the number of
times the algorithm must consult the oracle `f` in the worst case — is a
prerequisite for claiming any quantum speedup. The original Deutsch
algorithm [1] addressed the single-bit case ($n=1$) and Deutsch and Jozsa
[2] generalised it to $n$ inputs. A quantum solution decides the class with
a single query for any $n$, whereas the deterministic classical lower bound
grows as $2^{n-1} + 1$ in the worst case [3, 4].

For our case of $n = 4$ Boolean inputs, this means the worst-case classical
query count is $2^{4-1} + 1 = 9$ calls to `f`. The next sub-section derives
this bound; the implementation that follows uses early termination so that
many inputs (most balanced functions) require far fewer than 9 calls in
practice.

The framing "potential advantage of quantum computing" in the problem
statement refers to the broader debate around quantum supremacy and
practical quantum advantage, surveyed accessibly by Preskill in the
*Quanta Magazine* interview cited in the problem [5]. Deutsch–Jozsa is a
clean demonstration of the *separation* between deterministic classical
and exact quantum query complexity, even though the problem itself has no
known practical application — a point Nielsen and Chuang make explicitly
[3, sec. 1.4.4].

### References

[1] D. Deutsch, "Quantum theory, the Church–Turing principle and the
universal quantum computer," *Proceedings of the Royal Society A*,
vol. 400, no. 1818, pp. 97–117, 1985.
https://doi.org/10.1098/rspa.1985.0070

[2] D. Deutsch and R. Jozsa, "Rapid solution of problems by quantum
computation," *Proceedings of the Royal Society A*, vol. 439, no. 1907,
pp. 553–558, 1992. https://doi.org/10.1098/rspa.1992.0167

[3] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University
Press, 2010, sec. 1.4.4.

[4] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum
algorithms revisited," *Proceedings of the Royal Society A*, vol. 454,
no. 1969, pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

[5] J. Preskill, interviewed by K. Hartnett, "John Preskill explains
quantum supremacy," *Quanta Magazine*, 2 October 2019.
https://www.quantamagazine.org/john-preskill-explains-quantum-supremacy-20191002/

In [20]:
### Worst-case query count: a derivation

Suppose we query `f` on a sequence of distinct inputs $x_1, x_2, \dots$ and
observe the outputs $f(x_1), f(x_2), \dots$. Each query reveals one bit of
information. We want the smallest number $k$ such that, after $k$ queries
in the worst case, we can decide *with certainty* whether `f` is constant
or balanced.

**Lower bound: $2^{n-1}+1$ queries are necessary.**

Imagine an adversary who is allowed to choose `f` *after* seeing our queries
(as long as the choice is consistent with the answers it has already
given). Suppose we make only $2^{n-1}$ queries and the adversary returns
the same value $b$ to every one of them. The observations are consistent
with two possibilities:

- the constant function $f(x) = b$, and
- a balanced function that returns $b$ on the $2^{n-1}$ inputs we queried
  and $\lnot b$ on the $2^{n-1}$ inputs we did not.

Both are valid promise functions, so we cannot distinguish them from $2^{n-1}$
queries alone. Therefore at least $2^{n-1}+1$ queries are required in the
worst case [1, sec. 1.4.4].

**Upper bound: $2^{n-1}+1$ queries are sufficient.**

If we have queried $2^{n-1}+1$ distinct inputs and they all returned the same
value, then a balanced function (which returns each value on exactly
$2^{n-1}$ of the $2^n$ inputs) is impossible — `f` must be constant.
Conversely, if among any $k \le 2^{n-1}+1$ queries we ever observe two
*different* values, then `f` cannot be constant, and by the promise it must
be balanced. So $2^{n-1}+1$ queries always suffice.

**For $n = 4$.**

$$
2^{n-1} + 1 = 2^{3} + 1 = 9.
$$

So in the worst case our classical algorithm needs **9 calls** to `f` to be
100% certain of its class. The expected number is far smaller in practice:
balanced functions reveal themselves on the first disagreeing query, which
typically happens within the first few calls. We quantify this empirically
in Step 6.

This $\Theta(2^{n})$ classical lower bound, contrasted with the single
quantum query of Deutsch–Jozsa, is the canonical first example of an
*exact* (zero-error) query separation between classical and quantum
computation [2].

### References

[1] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.4.

[2] H. Buhrman and R. de Wolf, "Complexity measures and decision tree
complexity: a survey," *Theoretical Computer Science*, vol. 288, no. 1,
pp. 21–43, 2002. https://doi.org/10.1016/S0304-3975(01)00144-X

SyntaxError: invalid syntax (1898819838.py, line 3)

In [ ]:
### Implementation strategy

The proof in the previous section gives us the algorithm directly. We
enumerate inputs to `f` one at a time and stop as soon as either of the
two terminating conditions is met:

1. **We see two different outputs.** The function cannot be constant, so
   by the Deutsch–Jozsa promise it must be **balanced**. We can return
   immediately, often after only 2 queries.

2. **We have queried $2^{n-1}+1 = 9$ distinct inputs and they all agree.**
   A balanced function would have produced disagreement by now (it agrees
   on at most $2^{n-1} = 8$ inputs), so the function must be **constant**.

Two further design choices are worth flagging:

- **Query order.** Any order of distinct inputs is correct, but we use the
  natural lexicographic order over $\{0,1\}^4$ produced by `itertools.product`.
  This makes the behaviour deterministic and reproducible — running
  `determine_constant_balanced` twice on the same `f` will issue the
  same queries in the same sequence — which matters for the call-count
  measurements in Step 6.
- **Counting calls without trusting the function.** We wrap `f` in a small
  counter so the demonstration in Step 6 can observe the *actual* number
  of times `f` was invoked, rather than relying on a number reported by
  the algorithm itself. This separation lets the classifier stay clean
  (no instrumentation in its body) while still giving us empirical data.

The implementation in the next cell follows this strategy directly. It
returns as soon as the answer is decidable and never makes more than 9
calls.